# skforecast-ai — Demo Phase 1 & 2

This notebook demonstrates the functionality implemented so far:
- **Phase 1**: Package skeleton & Pydantic schemas (`DataProfile`, `ForecastPlan`)
- **Phase 2**: Data Profiler (`create_data_profile`)

## Phase 1 — Schemas

In [1]:
import skforecast_ai

print(f"skforecast-ai version: {skforecast_ai.__version__}")

skforecast-ai version: 0.1.0


In [2]:
from skforecast_ai.schemas import DataProfile, ForecastPlan

# Create a minimal DataProfile
profile = DataProfile(
    n_observations=365,
    n_series=1,
    index_type="datetime",
    target="y",
)
profile

DataProfile(n_observations=365, n_series=1, index_type='datetime', frequency=None, target='y', date_column=None, series_id_column=None, exog_columns=[], categorical_exog=[], missing_values={}, inferred_seasonalities=[], warnings=[])

In [3]:
# Create a ForecastPlan with all fields
plan = ForecastPlan(
    task_type="single_series",
    forecaster="ForecasterRecursive",
    estimator="LGBMRegressor",
    horizon=24,
    frequency="h",
    lags=[1, 2, 3, 24],
    metric="mean_absolute_error",
    backtesting_strategy="TimeSeriesFold",
    interval_method="bootstrapping",
    use_exog=True,
    data_requirements=["At least 2 complete seasonal cycles"],
    warnings=[],
    rationale="Single hourly series with exogenous variables.",
)
plan

ForecastPlan(task_type='single_series', forecaster='ForecasterRecursive', estimator='LGBMRegressor', horizon=24, frequency='h', lags=[1, 2, 3, 24], metric='mean_absolute_error', backtesting_strategy='TimeSeriesFold', interval_method='bootstrapping', use_exog=True, data_requirements=['At least 2 complete seasonal cycles'], warnings=[], rationale='Single hourly series with exogenous variables.')

In [4]:
# JSON roundtrip
json_str = plan.model_dump_json(indent=2)
print(json_str)

{
  "task_type": "single_series",
  "forecaster": "ForecasterRecursive",
  "estimator": "LGBMRegressor",
  "horizon": 24,
  "frequency": "h",
  "lags": [
    1,
    2,
    3,
    24
  ],
  "metric": "mean_absolute_error",
  "backtesting_strategy": "TimeSeriesFold",
  "interval_method": "bootstrapping",
  "use_exog": true,
  "data_requirements": [
    "At least 2 complete seasonal cycles"
  ],
  "warnings": [],
  "rationale": "Single hourly series with exogenous variables."
}


In [5]:
restored = ForecastPlan.model_validate_json(json_str)
assert restored == plan
print("JSON roundtrip OK")

JSON roundtrip OK


## Phase 2 — Data Profiler

### Single series — daily

In [6]:
import numpy as np
import pandas as pd

from skforecast_ai.profiling import create_data_profile

In [7]:
# Single daily series
df_daily = pd.DataFrame(
    {"y": np.random.default_rng(42).normal(100, 10, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)
df_daily.head()

,y
2023-01-01,103.047171
2023-01-02,89.600159
2023-01-03,107.504512
2023-01-04,109.405647
2023-01-05,80.489648


In [8]:
profile = create_data_profile(data=df_daily, target="y")
profile

DataProfile(n_observations=365, n_series=1, index_type='datetime', frequency='D', target='y', date_column=None, series_id_column=None, exog_columns=[], categorical_exog=[], missing_values={}, inferred_seasonalities=[7, 365], warnings=[])

### Single series — hourly with exogenous variables

In [9]:
rng = np.random.default_rng(42)
df_hourly = pd.DataFrame(
    {
        "sales": rng.normal(500, 50, 720),
        "temperature": np.tile(np.linspace(5, 35, 24), 30),
        "promo_budget": rng.uniform(100, 1000, 720),
        "holiday": ["no"] * 700 + ["yes"] * 20,
    },
    index=pd.date_range("2023-01-01", periods=720, freq="h"),
)
df_hourly.head()

,sales,temperature,promo_budget,holiday
2023-01-01 00:00:00,515.235854,5.000000,334.313689,no
2023-01-01 01:00:00,448.000795,6.304348,390.241397,no
2023-01-01 02:00:00,537.522560,7.608696,318.234568,no
2023-01-01 03:00:00,547.028236,8.913043,531.877060,no
2023-01-01 04:00:00,402.448241,10.217391,714.932522,no


In [10]:
profile = create_data_profile(data=df_hourly, target="sales")
profile

DataProfile(n_observations=720, n_series=1, index_type='datetime', frequency='h', target='sales', date_column=None, series_id_column=None, exog_columns=['temperature', 'promo_budget', 'holiday'], categorical_exog=['holiday'], missing_values={}, inferred_seasonalities=[24, 168], warnings=[])

### Multi-series — long format

In [11]:
dates = pd.date_range("2023-01-01", periods=100, freq="D")
df_multi = pd.DataFrame(
    {
        "date": np.tile(dates, 3),
        "series_id": np.repeat(["store_A", "store_B", "store_C"], 100),
        "sales": rng.normal(200, 30, 300),
        "promo": rng.choice([0, 1], 300),
    }
)
df_multi.head()

,date,series_id,sales,promo
0,2023-01-01,store_A,191.872156,0
1,2023-01-02,store_A,172.130325,1
2,2023-01-03,store_A,173.954197,0
3,2023-01-04,store_A,150.936103,1
4,2023-01-05,store_A,207.108458,0


In [12]:
profile = create_data_profile(
    data=df_multi,
    target="sales",
    date_column="date",
    series_id_column="series_id",
)
profile

DataProfile(n_observations=300, n_series=3, index_type='datetime', frequency=None, target='sales', date_column='date', series_id_column='series_id', exog_columns=['promo'], categorical_exog=[], missing_values={}, inferred_seasonalities=[], warnings=['Could not infer frequency from the datetime index. The series may have irregular spacing or gaps.'])

### Edge cases — warnings

In [13]:
# Short series (< 50 observations)
df_short = pd.DataFrame(
    {"y": np.arange(20, dtype=float)},
    index=pd.date_range("2023-01-01", periods=20, freq="D"),
)
profile = create_data_profile(data=df_short, target="y")
print("Warnings:", profile.warnings)

Warnings: ['Short series: only 20 observations. Results may be unreliable with fewer than 50 observations.']


In [14]:
# No datetime index
df_no_date = pd.DataFrame({"value": np.arange(100, dtype=float)})
profile = create_data_profile(data=df_no_date, target="value")
print(f"index_type: {profile.index_type}")
print(f"frequency:  {profile.frequency}")
print("Warnings:", profile.warnings)

index_type: range
frequency:  None
Warnings: ['No datetime index detected. Frequency inference and seasonality estimation are unavailable.']


In [15]:
# Missing values
values = np.arange(100, dtype=float)
values[10] = np.nan
values[20] = np.nan
values[30] = np.nan
df_missing = pd.DataFrame(
    {"target": values},
    index=pd.date_range("2023-01-01", periods=100, freq="D"),
)
profile = create_data_profile(data=df_missing, target="target")
print(f"missing_values: {profile.missing_values}")

missing_values: {'target': 3}
